# MedVision-AI Kaggle launcher

This notebook is intentionally thin. It synchronizes `main`, validates the Kaggle runtime and canonical checkpoint, then launches reproducible Stage 1 with safe auto-resume. Runtime artifacts live outside the repository at `/kaggle/working/medvision_outputs`.


In [1]:
# 1. Environment and canonical paths
import os
import platform
import subprocess
import sys
from pathlib import Path

os.environ['KERAS_BACKEND'] = 'tensorflow'
WORKING = Path('/kaggle/working')
REPO = WORKING / 'MedVision-AI'
RUNTIME_OUTPUT = WORKING / 'medvision_outputs'
REPO_URL = 'https://github.com/SwastikPandey1024/MedVision-AI.git'
STAGE1_CHECKPOINT = RUNTIME_OUTPUT / 'checkpoints' / 'densenet121_stage1_best.keras'

print('Python:', platform.python_version(), sys.executable)
print('Repository:', REPO)
print('Runtime artifacts:', RUNTIME_OUTPUT)
print('Canonical Stage 1 checkpoint:', STAGE1_CHECKPOINT)


Python: 3.12.13 /usr/bin/python3
Repository: /kaggle/working/MedVision-AI
Runtime artifacts: /kaggle/working/medvision_outputs
Canonical Stage 1 checkpoint: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage1_best.keras


In [2]:
# 2. Synchronize a fresh or existing Kaggle checkout; artifacts remain untouched.
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO), '--no-deps'], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Verified commit:', commit)


Cloning into '/kaggle/working/MedVision-AI'...


Obtaining file:///kaggle/working/MedVision-AI
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for medvision-ai (pyproject.toml): started
  Building editable for medvision-ai (pyproject.toml): finished with status 'done'
  Created wheel for medvision-ai: filename=medvision_ai-0.1.0a0-0.editable-py3-none-any.whl size=5465 sha256=46ed80a19dad7c2dddb38cde317b9bf7ae79f36de11de4f34a53cde87403459d
  Stored in directory: /tmp/pip-ephem-wheel-cache-ayp02zgx/wheels/36/e6/50/acd6ec13a8d518dbbfeb398fcb5ef565f202f51427d5d153e8
Successfully

In [3]:
# 3. Verify TensorFlow/Keras, GPU, dataset attachment, and any runtime checkpoint.
import tensorflow as tf
import keras

sys.path.insert(0, str(REPO / 'src'))
from medvision.data.dataset import find_dataset_root
from medvision.models.trainer import find_valid_resume_checkpoint

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__, '| Keras:', keras.__version__)
print('GPUs:', [gpu.name for gpu in gpus])
if not gpus:
    raise RuntimeError('GPU is required for --mode full; enable a Kaggle GPU accelerator.')

dataset_root = find_dataset_root()
labels = list(dataset_root.glob('*train_labels.csv'))
print('Dataset root:', dataset_root)
if not labels:
    raise FileNotFoundError('Attach the RSNA Pneumonia Detection Challenge dataset to this Kaggle notebook.')

valid_checkpoint = find_valid_resume_checkpoint(STAGE1_CHECKPOINT, 'densenet121')
if valid_checkpoint:
    print(f'Resume checkpoint accepted: {valid_checkpoint.path} ({valid_checkpoint.size_bytes / 2**20:.2f} MB, optimizer steps={valid_checkpoint.optimizer_iterations})')
else:
    print('No valid runtime checkpoint found: launcher will run a clean Stage 1 after preflight.')


TensorFlow: 2.20.0 | Keras: 3.13.2
GPUs: ['/physical_device:GPU:0', '/physical_device:GPU:1']
[2026-08-13 13:37:05] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
Dataset root: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 13:37:05] [INFO] [medvision.models.trainer:91] - AUTO-RESUME: no canonical checkpoint found at /kaggle/working/medvision_outputs/checkpoints/densenet121_stage1_best.keras
No valid runtime checkpoint found: launcher will run a clean Stage 1 after preflight.


In [4]:
# 4. Single reproducible launcher. It performs controlled preflight, auto-resumes
# the canonical valid checkpoint if present, preserves save_best_only, and verifies persistence.
subprocess.run([
    sys.executable, 'scripts/train.py',
    '--mode', 'full',
    '--stage', 'stage1',
    '--epochs', '5',
    '--batch-size', '64',
    '--mixed-precision',
    '--auto-resume',
], cwd=REPO, check=True)


[2026-08-13 13:37:10] [INFO] [medvision.train_script:320] - ===========================================================================
[2026-08-13 13:37:10] [INFO] [medvision.train_script:321] - MedVision-AI Controlled Cloud Training Engine
[2026-08-13 13:37:10] [INFO] [medvision.train_script:322] - ===========================================================================
[2026-08-13 13:37:10] [INFO] [medvision.train_script:323] - Execution Mode       : full
[2026-08-13 13:37:10] [INFO] [medvision.train_script:324] - Model Architecture   : densenet121
[2026-08-13 13:37:10] [INFO] [medvision.train_script:325] - Target Stage         : stage1
[2026-08-13 13:37:10] [INFO] [medvision.train_script:326] - Max Epochs per Stage : 5
[2026-08-13 13:37:10] [INFO] [medvision.train_script:327] - Smoke Test Mode      : False
[2026-08-13 13:37:10] [INFO] [medvision.train_script:354] - ===========================================================================
[2026-08-13 13:37:10] [INFO] [medvision

I0000 00:00:1786628235.880240     101 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786628235.883347     101 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[2026-08-13 13:37:15] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 13:37:15] [INFO] [medvision.train_script:157] - REAL_RSNA_DATASET = YES
[2026-08-13 13:37:15] [INFO] [medvision.train_script:159] - Dataset Root Resolved: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 13:37:15] [INFO] [medvision.train_script:160] - Labels File Resolved : /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv
[2026-08-13 13:37:15] [INFO] [medvision.train_script:402] - Full mode selected: Resolving RSNA dataset root & data pipelines...
[2026-08-13 13:37:15] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 13:37:15] [INFO] [medvision.train_script:422] - TFRecord shards not found at '/kaggle/working/MedVision-AI/data/processed/tfrecords'. Building real RSNA DICOM datasets direc

[2026-08-13 13:39:10] [INFO] [medvision.models.trainer:848] - INPUT       : dtype=<dtype: 'float32'> | shape=(32, 224, 224, 3) | min=0.0000 | max=1.0000 | mean=0.5522 | finite=True
[2026-08-13 13:39:10] [INFO] [medvision.models.trainer:851] - LABELS      : dtype=<dtype: 'float32'> | unique=[0.0, 1.0] | pos=7 | neg=25 | finite=True
[2026-08-13 13:39:10] [INFO] [medvision.models.trainer:854] - PREDICTIONS : dtype=<dtype: 'float32'> | shape=(32, 1) | min=0.0566 | max=0.9675 | finite=True
[2026-08-13 13:39:10] [INFO] [medvision.models.trainer:857] - RAW BCE     : value=1.1349 | finite=True | dtype=float32
[2026-08-13 13:39:10] [INFO] [medvision.models.trainer:860] - WEIGHTED BCE: value=1.2021 | finite=True | dtype=float32
[2026-08-13 13:39:10] [INFO] [medvision.models.trainer:863] - GRADIENTS   : norm=16.6819 | None_count=0 | range=[-1.6719e+00, 1.4883e+00] | finite=True
[2026-08-13 13:39:10] [INFO] [medvision.models.trainer:866] - OPTIMIZER   : class=LossScaleOptimizer | lr=9.999999747378

I0000 00:00:1786628416.650268     689 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786628416.652817     689 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[2026-08-13 13:40:16] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 13:40:16] [INFO] [medvision.train_script:157] - REAL_RSNA_DATASET = YES
[2026-08-13 13:40:16] [INFO] [medvision.train_script:159] - Dataset Root Resolved: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 13:40:16] [INFO] [medvision.train_script:160] - Labels File Resolved : /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv
[2026-08-13 13:40:16] [INFO] [medvision.train_script:402] - Full mode selected: Resolving RSNA dataset root & data pipelines...
[2026-08-13 13:40:16] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 13:40:16] [INFO] [medvision.train_script:422] - TFRecord shards not found at '/kaggle/working/MedVision-AI/data/processed/tfrecords'. Building real RSNA DICOM datasets direc

CompletedProcess(args=['/usr/bin/python3', 'scripts/train.py', '--mode', 'full', '--stage', 'stage1', '--epochs', '5', '--batch-size', '64', '--mixed-precision', '--auto-resume'], returncode=0)

In [5]:
# 5. Final artifact inventory and summary.
from medvision.models.trainer import verify_checkpoint_persistence

verify_checkpoint_persistence(str(STAGE1_CHECKPOINT))
for path in sorted(RUNTIME_OUTPUT.rglob('*')):
    if path.is_file():
        print(f'{path.relative_to(RUNTIME_OUTPUT)} | {path.stat().st_size / 2**20:.2f} MB')
print('Kaggle runtime artifacts are session-local. Download or explicitly export the checkpoint and metrics if you need them after a runtime reset.')



CHECKPOINT PERSISTENCE: PASS
CHECKPOINT PATH: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage1_best.keras
CHECKPOINT SIZE MB: 33.33 MB

[2026-08-13 14:03:21] [INFO] [medvision.models.trainer:170] - CHECKPOINT PERSISTENCE: PASS | PATH: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage1_best.keras | SIZE: 33.33 MB
checkpoints/densenet121_stage1_best.keras | 33.33 MB
logs/tensorboard/densenet121_stage1_full/train/events.out.tfevents.1786628433.338ce0381cba.689.0.v2 | 2.74 MB
logs/tensorboard/densenet121_stage1_full/validation/events.out.tfevents.1786628686.338ce0381cba.689.1.v2 | 0.01 MB
metrics/densenet121_stage1_full_history.csv | 0.00 MB
Kaggle runtime artifacts are session-local. Download or explicitly export the checkpoint and metrics if you need them after a runtime reset.


In [6]:
from pathlib import Path
import zipfile

root = Path("/kaggle/working/medvision_outputs")
zip_path = Path("/kaggle/working/medvision_stage1_artifacts.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(root))

print("=" * 70)
print("STAGE 1 ARTIFACT PACKAGE CREATED")
print("=" * 70)
print("ZIP:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / 1024**2, 2))
print("=" * 70)

STAGE 1 ARTIFACT PACKAGE CREATED
ZIP: /kaggle/working/medvision_stage1_artifacts.zip
Size MB: 30.71
